# Lecture 09 - Solutions to Part B (Python Applications)

This notebook gives worked Python solutions for Exercises 10-15. The focus is simple linear regression, correlation, fitted values, residuals, goodness of fit, slope uncertainty, interpolation, extrapolation, and model shape.

The official exercise workbook is `data_lecture9_exercises.xlsx`. It is expected in the lecture `data` folder. Use sheet `data1` for Exercises 10-14, and sheets `data2`, `data3`, and `data4` for Exercise 15. If you store the workbook elsewhere, change only `data_path`.


## Setup

Run this cell first. It loads the official exercise workbook and defines small helper functions for simple regression.


In [1]:
import numpy as np
import pandas as pd
from math import sqrt

try:
    import statsmodels.api as sm
except ImportError:
    sm = None

pd.set_option("display.float_format", "{:.6f}".format)

data_path = "../data/"  # Use your data here. Keep the final slash.

regression_file = data_path + "data_lecture9_exercises.xlsx"

def simple_regression(y, x):
    """Return ordinary least-squares results for y = intercept + slope*x."""
    y = pd.Series(y).astype(float).reset_index(drop=True)
    x = pd.Series(x).astype(float).reset_index(drop=True)
    n = len(y)
    x_bar = x.mean()
    y_bar = y.mean()
    sxx = ((x - x_bar) ** 2).sum()
    sxy = ((x - x_bar) * (y - y_bar)).sum()
    slope = sxy / sxx
    intercept = y_bar - slope * x_bar
    fitted = intercept + slope * x
    residuals = y - fitted
    sum_squared_residuals = (residuals ** 2).sum()
    corr = x.corr(y)
    r2 = corr ** 2
    residual_variance = sum_squared_residuals / (n - 2)
    residual_se = sqrt(residual_variance)
    se_slope = sqrt(residual_variance / sxx)
    se_intercept = sqrt(residual_variance * (1/n + x_bar**2 / sxx))
    return {
        "n": n,
        "intercept": intercept,
        "slope": slope,
        "fitted": fitted,
        "residuals": residuals,
        "sum_squared_residuals": sum_squared_residuals,
        "r2": r2,
        "corr": corr,
        "residual_se": residual_se,
        "se_slope": se_slope,
        "se_intercept": se_intercept,
        "t_slope_zero": slope / se_slope,
    }

def regression_summary(y, x):
    out = simple_regression(y, x)
    return pd.Series({
        "n": out["n"],
        "intercept": out["intercept"],
        "slope": out["slope"],
        "correlation": out["corr"],
        "r_squared": out["r2"],
        "residual_se": out["residual_se"],
        "se_slope": out["se_slope"],
        "t_slope_zero": out["t_slope_zero"],
    })


---

## Exercise 10 - Load the Regression Workbook


In [2]:
data1 = pd.read_excel(regression_file, sheet_name="data1")

data1.head()

   Cost     Revenue
0    10  -36.106973
1    13  -80.692584
2    21 -160.349575
3    17 -113.562310
4    16 -109.126017

In [3]:
summary_10 = pd.Series({
    "n": len(data1),
    "mean_cost": data1["Cost"].mean(),
    "mean_revenue": data1["Revenue"].mean(),
    "min_cost": data1["Cost"].min(),
    "max_cost": data1["Cost"].max(),
})

summary_10


n                43.000000
mean_cost        16.627907
mean_revenue   -104.012260
min_cost         10.000000
max_cost         25.000000
dtype: float64

---

## Exercise 11 - Estimate a Simple Regression by Formula


In [4]:
result_11 = simple_regression(y=data1["Revenue"], x=data1["Cost"])

pd.Series({
    "slope": result_11["slope"],
    "intercept": result_11["intercept"],
    "correlation": result_11["corr"],
})


slope         -8.192732
intercept     32.215720
correlation   -0.926012
dtype: float64

In [5]:
data1_solution = data1.copy()
data1_solution["fitted_revenue"] = result_11["fitted"]
data1_solution["residual"] = result_11["residuals"]

data1_solution.head()


   Cost     Revenue  fitted_revenue   residual
0    10  -36.106973      -49.711596  13.604623
1    13  -80.692584      -74.289791  -6.402793
2    21 -160.349575     -139.831645 -20.517931
3    17 -113.562310     -107.060718  -6.501592
4    16 -109.126017      -98.867986 -10.258031

---

## Exercise 12 - Goodness of Fit


In [6]:
if sm is not None:
    X = sm.add_constant(data1["Cost"])
    model_12 = sm.OLS(data1["Revenue"], X).fit()
    r_squared_from_output = model_12.rsquared
else:
    model_12 = None
    r_squared_from_output = result_11["r2"]

r = result_11["corr"]
r_squared_from_correlation = r ** 2

pd.Series({
    "R^2 from regression output": r_squared_from_output,
    "r^2 from squared correlation": r_squared_from_correlation,
})


R^2 from regression output     0.857498
r^2 from squared correlation   0.857498
dtype: float64

In simple regression with an intercept, the coefficient of determination \(R^2\) is equal to \(r^2\), the squared correlation between the dependent and independent variables. Small numerical differences can occur only because of rounding. A value close to 1 means that the fitted line explains a large share of the variation in the dependent variable; a value close to 0 means that it explains little. This is not, by itself, proof of causality or proof that the model is useful for every prediction task.


---

## Exercise 13 - Slope Uncertainty


In [7]:
if model_12 is not None:
    ci = model_12.conf_int().loc["Cost"]
    slope_uncertainty = pd.Series({
        "slope": model_12.params["Cost"],
        "se_slope": model_12.bse["Cost"],
        "t_slope_zero": model_12.tvalues["Cost"],
        "ci95_low": ci.iloc[0],
        "ci95_high": ci.iloc[1],
    })
else:
    critical = 1.96
    slope_ci = (
        result_11["slope"] - critical * result_11["se_slope"],
        result_11["slope"] + critical * result_11["se_slope"],
    )
    slope_uncertainty = pd.Series({
        "slope": result_11["slope"],
        "se_slope": result_11["se_slope"],
        "t_slope_zero": result_11["t_slope_zero"],
        "ci95_low": slope_ci[0],
        "ci95_high": slope_ci[1],
    })

slope_uncertainty


slope           -8.192732
se_slope         0.521593
t_slope_zero   -15.707144
ci95_low        -9.215053
ci95_high       -7.170410
dtype: float64

The approximate confidence interval summarizes uncertainty about the slope estimate. If the interval does not include zero, the sample gives strong evidence of a nonzero linear relationship under the model assumptions.

---

## Exercise 14 - Interpolation and Extrapolation


In [8]:
min_cost = data1["Cost"].min()
max_cost = data1["Cost"].max()

prediction_points = pd.DataFrame({"Cost": [min_cost, 15, 20, max_cost, 80]})
prediction_points["predicted_revenue"] = result_11["intercept"] + result_11["slope"] * prediction_points["Cost"]
prediction_points["type"] = np.where(
    prediction_points["Cost"].between(min_cost, max_cost),
    "interpolation",
    "extrapolation"
)

prediction_points

   Cost  predicted_revenue           type
0    10         -49.711596  interpolation
1    15         -90.675255  interpolation
2    20        -131.638913  interpolation
3    25        -172.602571  interpolation
4    80        -623.202812  extrapolation

The prediction for cost 80 is extrapolation because it is outside the observed cost range. It relies on the assumption that the fitted linear pattern continues far beyond the data used to estimate the model.

---

## Exercise 15 - Nonlinear Data and Linear Fit


In [9]:
sheet_specs = {
    "data2": ("Y", "X"),
    "data3": ("Y", "X"),
    "data4": ("Value", "Time"),
}

rows = []
for sheet, (y_col, x_col) in sheet_specs.items():
    df = pd.read_excel(regression_file, sheet_name=sheet)
    out = simple_regression(df[y_col], df[x_col])
    rows.append({
        "sheet": sheet,
        "dependent": y_col,
        "independent": x_col,
        "intercept": out["intercept"],
        "slope": out["slope"],
        "r_squared": out["r2"],
        "correlation": out["corr"],
    })

nonlinear_summary = pd.DataFrame(rows)
nonlinear_summary

   sheet dependent independent  intercept     slope  r_squared  correlation
0  data2         Y           X 833.245514 10.033173   0.076516     0.276616
1  data3         Y           X   3.665905 -2.577661   0.148013    -0.384724
2  data4     Value        Time 158.163883 -6.488489   0.776559    -0.881226

A high or low \(R^2\) does not prove that a straight-line model is appropriate. A nonlinear relationship can have a misleading linear summary, and residual patterns should be inspected. Among these sheets, a dataset generated by a curved relationship is the most problematic for a straight-line interpretation.
